# Transformer encoder for classification

This is the code for the transformer encoder model used for the project.


In [2]:
import tensorflow as tf
from keras import layers
import keras
from datasets import load_dataset
import keras.ops as ops

ds = load_dataset("google/civil_comments")

print(ds)
print(ds["train"][1])

text_vectorization = keras.layers.TextVectorization(
    max_tokens=20000,
    output_mode="int",
    output_sequence_length=300,
)

train_text = ds["train"]["text"]
text_vectorization.adapt(train_text)

label_names = [
    'toxicity',
    'severe_toxicity',
    'obscene',
    'threat',
    'insult',
    'identity_attack',
    'sexual_explicit'
]

def format_example(text, labels):
    x = text_vectorization(text)
    x.set_shape([300])
    y = tf.stack(labels, axis=-1)
    return x, y

def gen(split):
    for x in ds[split]:
        yield x["text"], tuple(x[label] for label in label_names)

output_signature = (
    tf.TensorSpec(shape=(), dtype=tf.string),
    tuple(tf.TensorSpec(shape=(), dtype=tf.float32) for _ in label_names)
)
def preprocess(batch):
    text = text_vectorization(batch["text"]) 
    labels = tf.stack([tf.stack([batch[label][i] for label in label_names], axis=-1)
                       for i in range(len(batch["text"]))])
    return {"input_ids": text, "labels": labels}

ds = ds.map(preprocess, batched=True)

def make_dataset(split): 
    return ds[split].to_tf_dataset(
        columns=["input_ids"],
        label_cols=["labels"],
        batch_size=32,
        shuffle=(split == "train"),
    ).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset("train")
val_ds = make_dataset("validation")
test_ds = make_dataset("test")


class PositionalEmbedding(keras.layers.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)

        self.token_embeddings = keras.layers.Embedding(
            input_dim=input_dim, output_dim=output_dim
        )
        self.position_embeddings = keras.layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim
        )
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.supports_masking = True

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_positions = self.position_embeddings(positions)
        embedded_tokens = self.token_embeddings(inputs)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs,0)

    def get_config(self):
        config = super().get_config()
        config.update({
            "output_dim": self.output_dim,
            "sequence_length": self.sequence_length,
            "input_dim": self.input_dim,
        })
        return config

class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.bool)
            mask = tf.tile(mask, [1, tf.shape(inputs)[1], 1])
        attn_output = self.attention(inputs, inputs, attention_mask=mask)
        out1 = self.layernorm1(inputs + attn_output)
        proj_output = self.dense_proj(out1)
        return self.layernorm2(out1 + proj_output)

inputs = keras.Input(shape=(None,), dtype="int64")

x = PositionalEmbedding(300, 20000, 128)(inputs)
x = TransformerEncoder(128, 128, 4)(x)
x = layers.GlobalMaxPooling1D()(x)

outputs = layers.Dense(len(label_names), activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(multi_label=True)]
)

model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint(
        "transformer_encoder_model.keras",
        save_best_only=True
    )
]

model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)

model = keras.models.load_model(
    "transformer_encoder_model.keras",
    custom_objects={
        "TransformerEncoder": TransformerEncoder,
        "PositionalEmbedding": PositionalEmbedding
    }
)

print(f"Test auc: {model.evaluate(test_ds)[1]:.3f}")

DatasetDict({
    train: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 1804874
    })
    validation: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 97320
    })
    test: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 97320
    })
})
{'text': "Thank you!! This would make my life a lot less anxiety-inducing. Keep it up, and don't let anyone get in your way!", 'toxicity': 0.0, 'severe_toxicity': 0.0, 'obscene': 0.0, 'threat': 0.0, 'insult': 0.0, 'identity_attack': 0.0, 'sexual_explicit': 0.0}


Parameter 'function'=<function preprocess at 0x000002BB009DEC10> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.
Map: 100%|██████████████████████████████████████████████████████████████| 97320/97320 [00:12<00:00, 7832.14 examples/s]
C:\Users\vladd\AppData\Local\Programs\Python\Python39\lib\site-packages\datasets\arrow_dataset.py:405: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labe

C:\Users\vladd\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\layer.py:965: UserWarning: Layer 'global_max_pooling1d' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ positional_embedding          │ (None, None, 128)         │       2,598,400 │ input_layer[0][0]          │
│ (PositionalEmbedding)         │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal (NotEqual)          │ (None, None)              │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ transformer_encoder           │ (None, None, 128)         │         297,344 │ positional_embedding[0][0… │
│ (TransformerEncoder)          │                           │                 │ not_equal[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_max_pooling1d          │ (None, 128)               │               0 │ transformer_encoder[0][0]  │
│ (GlobalMaxPooling1D)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 7)                 │             903 │ global_max_pooling1d[0][0] │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

...

 Trainable params: 2,896,647 (11.05 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
56403/56403 ━━━━━━━━━━━━━━━━━━━━ 22840s 405ms/step - auc: 0.8160 - loss: 0.0996 - val_auc: 0.8438 - val_loss: 0.0928
Epoch 2/5
56403/56403 ━━━━━━━━━━━━━━━━━━━━ 24223s 429ms/step - auc: 0.8441 - loss: 0.0942 - val_auc: 0.8383 - val_loss: 0.0926
Epoch 3/5
56403/56403 ━━━━━━━━━━━━━━━━━━━━ 26709s 474ms/step - auc: 0.8476 - loss: 0.0935 - val_auc: 0.8358 - val_loss: 0.0925
Epoch 4/5
56403/56403 ━━━━━━━━━━━━━━━━━━━━ 28536s 506ms/step - auc: 0.8502 - loss: 0.0930 - val_auc: 0.8433 - val_loss: 0.0929
Epoch 5/5
56403/56403 ━━━━━━━━━━━━━━━━━━━━ 29725s 527ms/step - auc: 0.8525 - loss: 0.0925 - val_auc: 0.8409 - val_loss: 0.0924


C:\Users\vladd\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'positional_embedding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
C:\Users\vladd\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'transformer_encoder', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


3042/3042 ━━━━━━━━━━━━━━━━━━━━ 611s 201ms/step - auc: 0.8435 - loss: 0.0937
Test auc: 0.842


Saving the model and vocabulary here:

In [10]:
import tensorflow as tf
import keras
from keras import layers
import keras.ops as ops
from datasets import load_dataset
import pickle

ds = load_dataset("google/civil_comments")

text_vectorization = keras.layers.TextVectorization(
    max_tokens=20000,
    output_mode="int",
    output_sequence_length=300,
)

train_text = ds["train"]["text"]
text_vectorization.adapt(train_text)

class PositionalEmbedding(keras.layers.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)

        self.token_embeddings = keras.layers.Embedding(
            input_dim=input_dim, output_dim=output_dim
        )
        self.position_embeddings = keras.layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim
        )
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.supports_masking = True

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_positions = self.position_embeddings(positions)
        embedded_tokens = self.token_embeddings(inputs)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs,0)

    def get_config(self):
        config = super().get_config()
        config.update({
            "output_dim": self.output_dim,
            "sequence_length": self.sequence_length,
            "input_dim": self.input_dim,
        })
        return config

class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.bool)
            mask = tf.tile(mask, [1, tf.shape(inputs)[1], 1])
        attn_output = self.attention(inputs, inputs, attention_mask=mask)
        out1 = self.layernorm1(inputs + attn_output)
        proj_output = self.dense_proj(out1)
        return self.layernorm2(out1 + proj_output)

model = keras.models.load_model(
    "transformer_encoder_model.keras",
    custom_objects={
        "TransformerEncoder": TransformerEncoder,
        "PositionalEmbedding": PositionalEmbedding
    }
)

model.save("transformation_model.keras")

pickle.dump(text_vectorization.get_vocabulary(), open("vocab.pkl", "wb"))

C:\Users\vladd\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'positional_embedding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
C:\Users\vladd\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'transformer_encoder', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
C:\Users\vladd\AppData\Local\Programs\Python\Python39\lib\site-packages\ker